# GenAI, LLM Comparison, and Responsible AI

Owner: Member 6

This notebook completes the final project module by comparing the 01-05 verification pipeline with GenAI/LLM alternatives and by evaluating the system through responsible AI risks, limitations, scalability, and future work.

## Role in the Project

The first five modules build an interpretable misinformation verification pipeline: claim extraction, dataset EDA, entity linking, KG reasoning, and Bayesian final verdicts. This module asks whether that pipeline is responsible enough for a real-world misinformation setting and how it compares with a GenAI-only or RAG-based approach.

No live LLM call is made in this notebook. That is a deliberate responsible AI choice: the comparison should be reproducible and grounded in the project evidence, not in a one-off generated answer.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'LIAR_dataset').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODULE_DIR = PROJECT_ROOT / '06_GenAI_Responsible_AI'
sys.path.insert(0, str(MODULE_DIR))

from run_responsible_ai_analysis import run_responsible_ai_analysis

SUMMARY_PATH = MODULE_DIR / 'responsible_ai_summary.json'
ANALYSIS_PATH = MODULE_DIR / 'responsible_ai_analysis.md'

PROJECT_ROOT, SUMMARY_PATH, ANALYSIS_PATH

## Project Evidence Used

The comparison below uses concrete outputs from the earlier notebooks: the LIAR EDA summary, extraction handoff, entity-linking summary, KG reasoning summary, and Bayesian final verdict summary.

In [ ]:
results = run_responsible_ai_analysis(PROJECT_ROOT)
metrics = results['metrics']
metrics

## Evidence From Notebooks 01-05

A high-quality responsible AI analysis should not be generic. The table below links each project stage to the evidence it produced and explains what that evidence means for responsible deployment.

In [ ]:
results['module_evidence']

## Key Critical Finding

The pipeline is transparent but deliberately low-coverage. Entity linking and KG reasoning are the bottlenecks: many claims cannot be safely mapped to a precise KG fact, so the Bayesian module mostly returns `uncertain`. This is not a failure of the responsible AI design; it is an honest signal that the available evidence is insufficient.

In [ ]:
{
    'entity_link_coverage_percent': metrics['entity_link_coverage_percent'],
    'property_mapping_percent': metrics['property_mapping_percent'],
    'kg_unknown_percent': metrics['kg_unknown_percent'],
    'bayesian_uncertain_percent': metrics['bayesian_uncertain_percent'],
    'average_decision_confidence': metrics['average_decision_confidence'],
}

## GenAI/LLM Comparison

A GenAI-only fact-checker would likely provide an answer for nearly every claim. That can look stronger in a demo, but in misinformation verification the ability to answer is not the same as the ability to verify. The table compares the current pipeline, an unconstrained LLM-only approach, and a safer RAG hybrid.

In [ ]:
results['comparison_rows']

## Why LLM-Only Is Not Enough

An LLM-only verifier has useful language abilities: it can paraphrase claims, identify missing context, and produce readable explanations. However, it also introduces serious risks for AT3's real-world AI framing: hallucinated evidence, prompt sensitivity, weak reproducibility, training-data bias, and overconfident language. In this project, the correct role for GenAI is not to replace the pipeline, but to support evidence retrieval and explanation under strict constraints.

## Responsible AI Risk Register

The following register connects each risk to evidence from this project, the possible harm, current controls, and recommended improvements.

In [ ]:
results['risk_register']

## False Positives and False Negatives

False positives occur when true or nuanced claims are labelled misinformation. False negatives occur when false claims are not challenged. In the current project, the larger practical issue is not aggressive false positives because the Bayesian module mostly abstains. The larger coverage issue is that false claims may remain `uncertain` because KG evidence is missing. This is safer than calling them true, but it is not enough for a production fact-checking system.

In [ ]:
final_summary = json.loads((PROJECT_ROOT / '05_Bayesian_Inference' / 'final_verdict_summary.json').read_text())
final_summary.get('reference_bucket_alignment')

## Bias and Fairness Reflection

LIAR is a political dataset with concentrated speakers, topics, parties, and contexts. Speaker-history counts can be useful metadata, but they can also punish or favour speakers based on historical coverage rather than the current claim. The Bayesian module therefore uses speaker history only as a weak prior and keeps the original LIAR label separate as `reference_label` for evaluation rather than evidence.

In [ ]:
dataset_summary = json.loads((PROJECT_ROOT / '02_Problem_Dataset_EDA' / 'dataset_summary.json').read_text())
{
    'top_speakers': dataset_summary['top_speakers'][:5],
    'top_subjects': dataset_summary['top_subjects'][:5],
    'missing_values': {
        'speaker_job': dataset_summary['missing_values']['speaker_job'],
        'state': dataset_summary['missing_values']['state'],
        'context': dataset_summary['missing_values']['context'],
    },
}

## Scalability and Deployment

The current pipeline can scale technically, but responsible deployment requires more than speed. The extraction and Bayesian stages can run in batches. Entity linking and KG querying need caching, rate-limit handling, and better property coverage. LLM-only verification would be easy to prototype but harder to govern at scale because cost, reproducibility, hallucination risk, and prompt variation all become operational problems.

## Recommended Future Work

The strongest future direction is a hybrid system: use LLMs for claim decomposition, query generation, and readable summaries, but require retrieved sources, KG evidence, calibrated uncertainty, and human review for high-impact claims.

In [ ]:
results['future_work']

## Report Handoff

The runner writes a polished markdown analysis and report-ready CSV tables. These files can be used directly in the final AT3 report sections on theoretical justification, critical reflection, responsible AI, scalability, and future work.

In [ ]:
results['artifacts']

## Final Position

The responsible conclusion is that this project should not claim to be a complete automated fact-checker. It is better framed as an interpretable verification support system. Compared with a GenAI-only approach, it is more auditable and less prone to unsupported confident answers. Compared with a mature RAG system, it has lower evidence coverage. The best next version would combine this pipeline's traceability with retrieval-grounded LLM support and human oversight.